# Transformer for sequence learning

Unlike an MLP ans a CNN - a Transformer can model relationships between positions across the whole sequence. A Transformer encoder learns contextual representations of each position:

- what is at this position
- what is around it
- which other positions matter for it

So it is especially useful when:
- long-range dependencies matter
- interactions between positions matter
- combinations of motifs matter

For a sequence like:

```text
A A A U G C C
```

a Transformer encoder lets each position look at the others using self-attention.

## Simplified pipeline

```text
sequence
  ↓
token encoding / embedding
  ↓
positional encoding
  ↓
TransformerEncoder
  ↓
pool across sequence
  ↓
linear layer
  ↓
scalar prediction
```

## Input representation

First, convert nucleotides into integer tokens:

- `A → 0`
- `U → 1`
- `G → 2`
- `C → 3`

So the sequence `AAAUGCC` becomes:

```text
[0, 0, 0, 1, 2, 3, 3]
```

## Steps:

### 1. Tokenization

```text
AAAUGCC
  ↓
[0, 0, 0, 1, 2, 3, 3]
```

### 2. Embedding layer

```python
self.embedding = nn.Embedding(num_embeddings=4, embedding_dim=d_model)
```

This turns each token into a dense vector. For example:

```text
(1, 7) → (1, 7, 16)
```

### 3. Positional encoding

Transformers do not automatically know sequence order, so we add positional information. This can be:
- learned positional embeddings (this is what we use here; the model learns position representations from data)
- sinusoidal positional encoding (position information is given by a fixed mathematical pattern)


### 4. Transformer encoder

The encoder outputs contextualized representations for each position in the sequence.


### 5. Pooling

To get one prediction per sequence, we reduce across sequence length:

```text
(1, 7, 16) → (1, 16)
```

### 6. Final linear layer

```python
self.fc = nn.Linear(d_model, 1)
```

Shape:

```text
(1, 16) → (1, 1)
```

## Shape summary

```text
tokens:             (1, 7)
embedding:          (1, 7, 16)
+ positional emb:   (1, 7, 16)
transformer:        (1, 7, 16)
mean pool:          (1, 16)
linear:             (1, 1)
```

## Self-attention: Query, Key, Value

A common intuition is:
- Query: what patterns am I looking for?
- Key: what information do I contain?
- Value: what information should I pass along?

The similarity score is computed as:

$$
\text{attention score} = QK^T
$$

More completely:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

## Multi-head attention

Transformers usually run several attention mechanisms in parallel. For example, with 4 heads:
- head 1 - motif detection
- head 2 - GC-content patterns
- head 3 - long-range dependencies
- head 4 - positional effects

Example shapes:

```text
embedding         (1, 7, 16)
Q, K, V           (1, 7, 16)
attention matrix  (1, 7, 7)
output vectors    (1, 7, 16)
```

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

In [7]:
class SeqTransformer(nn.Module):
    def __init__(self, 
                 d_model=16, 
                 nhead=4, 
                 num_layers=2, 
                 max_len=100):
        super().__init__()

        self.embedding = nn.Embedding(4, d_model) # A,U,G,C -> vectors
        self.pos_embedding = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=64,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        # x shape: (batch, seq_len), integer tokens
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.embedding(x) + self.pos_embedding(positions)  # (batch, seq_len, d_model)
        x = self.transformer(x)                                # (batch, seq_len, d_model)
        x = x.mean(dim=1)                                      # (batch, d_model)
        x = self.fc(x)                                         # (batch, 1)
        return x

def encode_seq_as_tokens(seq):
    mapping = {"A": 0, "U": 1, "G": 2, "C": 3}
    return torch.tensor([mapping[n] for n in seq], dtype=torch.long)


In [8]:
seq = "AAAUGCC"
x = encode_seq_as_tokens(seq).unsqueeze(0)   # (1, 7)

model = SeqTransformer()
out = model(x)
score = out.item()
print(f"Score: {score:.2f}")

Score: -0.26


In [ ]:
# More input examples:
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [0.8, 0.3, 0.6]

X = torch.stack([encode_seq_as_tokens(seq) for seq in sequences])   # (3, 7)
Y = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)         # (3, 1)

model = SeqTransformer()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    
    model.train()

    optimizer.zero_grad()
    prediction = model(X)
    loss = loss_fn(prediction, Y)
    loss.backward()
    optimizer.step()

    print(f"epoch={epoch}, loss={loss.item():.3f}")

epoch=0, loss=1.165
epoch=1, loss=0.949
epoch=2, loss=0.697
epoch=3, loss=0.483
epoch=4, loss=0.324
epoch=5, loss=0.256
epoch=6, loss=0.183
epoch=7, loss=0.093
epoch=8, loss=0.102
epoch=9, loss=0.101


In [ ]:
# Predict! 
model.eval()
with torch.no_grad(): # do not track gradients for operations inside this block
    test_seq = encode_seq_as_tokens("AUGGCUACGU").unsqueeze(0)  # (1, seq_len)
    pred = model(test_seq)                                        # (1, 1)
    print(f"Predicted score: {pred.item():.3f}")

Predicted score: 0.652
